# FIFA 21 Data Cleaning (SDI Python Module)

Raw FIFA 21 player data: 18,979 players, 77 columns. Goal: turn messy text columns into clean, typed columns that are ready for analysis.

Cleaning steps:
1. Fill nulls
2. Rename and drop columns
3. Convert text columns to numbers (W/F, SM, IR, Hits, Weight, Height, Value, Wage, Release Clause)
4. Split Team & Contract into Team, Start Contract, End Contract, On Loan
5. Remove duplicates
6. Filter the age range
7. Save the clean file

In [21]:
import pandas as pd
import numpy as np

In [ ]:
# low_memory=False stops the mixed types warning on the Hits column
df = pd.read_csv('/content/fifa21_raw_data.csv', low_memory=False)
df.head()

In [24]:
df.shape

(18979, 77)

In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18979 entries, 0 to 18978
Data columns (total 77 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   photoUrl          18979 non-null  object
 1   LongName          18979 non-null  object
 2   playerUrl         18979 non-null  object
 3   Nationality       18979 non-null  object
 4   Positions         18979 non-null  object
 5   Name              18979 non-null  object
 6   Age               18979 non-null  int64 
 7   ↓OVA              18979 non-null  int64 
 8   POT               18979 non-null  int64 
 9   Team & Contract   18979 non-null  object
 10  ID                18979 non-null  int64 
 11  Height            18979 non-null  object
 12  Weight            18979 non-null  object
 13  foot              18979 non-null  object
 14  BOV               18979 non-null  int64 
 15  BP                18979 non-null  object
 16  Growth            18979 non-null  int64 
 17  Joined      

In [26]:
pd.set_option('display.max_rows', None)
df.isnull().sum()

,0
photoUrl,0
LongName,0
playerUrl,0
Nationality,0
Positions,0
Name,0
Age,0
↓OVA,0
POT,0
Team & Contract,0


## 1. Nulls

In [27]:
# Loan Date End is the only column with nulls (17,966), because most players are not on loan
# fill only that column so nulls in other columns are never hidden by accident
df_new = df.copy()
df_new['Loan Date End'] = df_new['Loan Date End'].fillna('No Loan')
df_new['Loan Date End'].value_counts().head()

,count
Loan Date End,
No Loan,17966
30-Jun-21,770
31-Dec-20,100
31-May-21,30
31-Dec-21,22


## 2. Rename and drop columns

In [28]:
df_new = df_new.rename(columns={'↓OVA': 'Overall Rating'})

# photo and player URLs are not useful for analysis
df_new = df_new.drop(columns=['photoUrl', 'playerUrl'])
df_new.columns

Index(['LongName', 'Nationality', 'Positions', 'Name', 'Age', 'Overall Rating',
       'POT', 'Team & Contract', 'ID', 'Height', 'Weight', 'foot', 'BOV', 'BP',
       'Growth', 'Joined', 'Loan Date End', 'Value', 'Wage', 'Release Clause',
       'Attacking', 'Crossing', 'Finishing', 'Heading Accuracy',
       'Short Passing', 'Volleys', 'Skill', 'Dribbling', 'Curve',
       'FK Accuracy', 'Long Passing', 'Ball Control', 'Movement',
       'Acceleration', 'Sprint Speed', 'Agility', 'Reactions', 'Balance',
       'Power', 'Shot Power', 'Jumping', 'Stamina', 'Strength', 'Long Shots',
       'Mentality', 'Aggression', 'Interceptions', 'Positioning', 'Vision',
       'Penalties', 'Composure', 'Defending', 'Marking', 'Standing Tackle',
       'Sliding Tackle', 'Goalkeeping', 'GK Diving', 'GK Handling',
       'GK Kicking', 'GK Positioning', 'GK Reflexes', 'Total Stats',
       'Base Stats', 'W/F', 'SM', 'A/W', 'D/W', 'IR', 'PAC', 'SHO', 'PAS',
       'DRI', 'DEF', 'PHY', 'Hits'],
      dtype

## 3. Text columns to numbers

In [29]:
# W/F, SM and IR are star ratings like '4 ★' or '4★', so keep the first character and make it a number
for col in ['W/F', 'SM', 'IR']:
    df_new[col] = df_new[col].astype(str).str.strip().str[0].astype(int)

df_new[['W/F', 'SM', 'IR']].describe()

,W/F,SM,IR
count,18979.000000,18979.000000,18979.00000
mean,2.940513,2.365035,1.09168
std,0.668449,0.766192,0.36182
min,1.000000,1.000000,1.00000
25%,3.000000,2.000000,1.00000
50%,3.000000,2.000000,1.00000
75%,3.000000,3.000000,1.00000
max,5.000000,5.000000,5.00000


In [30]:
# Hits looks like '\n372' or '\n1.6K', so strip the newlines and turn 'K' into thousands
def to_number(x):
    x = str(x).strip()
    if x.endswith('K'):
        return float(x[:-1]) * 1000
    return float(x)

df_new['Hits'] = df_new['Hits'].apply(to_number)
df_new['Hits'].describe()

,Hits
count,18979.000000
mean,16.934085
std,73.203131
min,1.000000
25%,2.000000
50%,3.000000
75%,10.000000
max,4500.000000


In [31]:
# Weight looks like '159lbs', so remove 'lbs' (works for 2 or 3 digit weights)
df_new['Weight'] = df_new['Weight'].str.replace('lbs', '', regex=False).astype(float)

# Height looks like 5'7", so convert feet and inches to total inches
feet_inches = df_new['Height'].str.extract(r"(\d+)'(\d+)")
df_new['Height'] = feet_inches[0].astype(int) * 12 + feet_inches[1].astype(int)

print('Average weight (lbs):', round(df_new['Weight'].mean(), 1))
print('Average height (inches):', round(df_new['Height'].mean(), 1))

Average weight (lbs): 165.4
Average height (inches): 71.3


In [32]:
# Value, Wage and Release Clause look like '€67.5M', '€560K' or '€500', so convert them to plain euro amounts
def euros(x):
    x = str(x).replace('€', '').strip()
    if x.endswith('M'):
        return float(x[:-1]) * 1_000_000
    if x.endswith('K'):
        return float(x[:-1]) * 1_000
    return float(x)

for col in ['Value', 'Wage', 'Release Clause']:
    df_new[col] = df_new[col].apply(euros)

df_new[['Value', 'Wage', 'Release Clause']].describe()

,Value,Wage,Release Clause
count,1.897900e+04,18979.000000,1.897900e+04
mean,2.228953e+06,8796.129933,3.961734e+06
std,5.097614e+06,19537.282500,9.771817e+06
min,0.000000e+00,0.000000,0.000000e+00
25%,3.000000e+05,1000.000000,4.230000e+05
50%,6.500000e+05,3000.000000,1.000000e+06
75%,1.800000e+06,8000.000000,2.800000e+06
max,1.055000e+08,560000.000000,2.031000e+08


## 4. Team & Contract

In [33]:
# raw values look like '\n\n\n\nFC Barcelona\n2004 ~ 2021\n\n'
# loan players look different, e.g. 'Team\nJun 30, 2021 On Loan', so they get their own flag
team_contract = df_new['Team & Contract'].str.strip()
parts = team_contract.str.split('\n')

df_new['Team'] = parts.str[0].str.strip()
contract = parts.str[-1].str.strip()

df_new[['Start Contract', 'End Contract']] = contract.str.extract(r'(\d{4})\s*~\s*(\d{4})')
df_new[['Start Contract', 'End Contract']] = df_new[['Start Contract', 'End Contract']].apply(pd.to_numeric).astype('Int64')
df_new['On Loan'] = contract.str.contains('Loan')

df_new = df_new.drop(columns=['Team & Contract'])
df_new[['Team', 'Start Contract', 'End Contract', 'On Loan']].head()

,Team,Start Contract,End Contract,On Loan
0,FC Barcelona,2004,2021,False
1,Juventus,2018,2022,False
2,Atlético Madrid,2014,2023,False
3,Manchester City,2015,2023,False
4,Paris Saint-Germain,2017,2022,False


In [34]:
# check: every player should have a team, and only loan players should be missing contract years
print('Missing team:', (df_new['Team'] == '').sum())
print('No contract years, not on loan:', (df_new['End Contract'].isna() & ~df_new['On Loan']).sum())
print('Players on loan:', df_new['On Loan'].sum())

Missing team: 0
No contract years, not on loan: 238
Players on loan: 1013


## 5. Duplicates

In [ ]:
# ID is the unique player number, so check it as well as full duplicate rows
print('Duplicate rows:', df_new.duplicated().sum())
print('Duplicate player IDs:', df_new['ID'].duplicated().sum())

df_new = df_new.drop_duplicates(subset='ID', keep='first')
df_new.shape

## 6. Age range

In [35]:
# Decision: keep players aged 18 to 39 for the analysis.
# Players under 18 and 40+ are real players, not data errors, so this is a scope choice, not a cleaning fix.
# Print how many rows the decision removes so it is documented.
before = len(df_new)
df_new = df_new[(df_new['Age'] >= 18) & (df_new['Age'] < 40)]
print('Rows removed by age filter:', before - len(df_new))
df_new.shape

Rows removed by age filter: 351


(18628, 78)

## 7. Final check and save

In [36]:
df_new.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18628 entries, 0 to 18978
Data columns (total 78 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   LongName          18628 non-null  object 
 1   Nationality       18628 non-null  object 
 2   Positions         18628 non-null  object 
 3   Name              18628 non-null  object 
 4   Age               18628 non-null  int64  
 5   Overall Rating    18628 non-null  int64  
 6   POT               18628 non-null  int64  
 7   ID                18628 non-null  int64  
 8   Height            18628 non-null  int64  
 9   Weight            18628 non-null  float64
 10  foot              18628 non-null  object 
 11  BOV               18628 non-null  int64  
 12  BP                18628 non-null  object 
 13  Growth            18628 non-null  int64  
 14  Joined            18628 non-null  object 
 15  Loan Date End     18628 non-null  object 
 16  Value             18628 non-null  float64
 17

In [37]:
df_new.to_csv('/content/fifa21_clean.csv', index=False)